##### Copyright 2018 The TensorFlow Hub Authors.

Licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Copyright 2018 The TensorFlow Hub Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Classify Flowers with Transfer Learning


## Setup


Load-in

In [ ]:
!pip install jupyter ipywidgets

import os
import tensorflow as tf
import tensorflow_datasets as tfds

# Detect and initialize the TPU
tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # Automatically detects the connected TPU
tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)

# Use TPUStrategy for distributed training on TPUs
strategy = tf.distribute.TPUStrategy(tpu)
print(f'Number of devices: {strategy.num_replicas_in_sync}')

Processing + Hyperparameters

In [10]:
# Load dataset
(train_dataset, test_dataset), dataset_info = tfds.load(
    'oxford_flowers102',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = dataset_info.features['label'].num_classes

# Constants and hyperparameters
INITIAL_LEARNING_RATE = 0.01
TRAIN_BATCH_SIZE = 1024  # Consider increasing this further for TPUs if your dataset allows
NUM_EPOCHS = 2000
VAL_FREQ = 200

# Preprocessing
def preprocess_for_vgg16(img, label):
    img = tf.image.resize(img, (224, 224))
    img = tf.keras.applications.vgg16.preprocess_input(img)
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label

AUTOTUNE = tf.data.experimental.AUTOTUNE
train_dataset = (train_dataset
                 .map(preprocess_for_vgg16, num_parallel_calls=AUTOTUNE)
                 .cache()
                 .shuffle(buffer_size=1000)
                 .batch(TRAIN_BATCH_SIZE)
                 .prefetch(AUTOTUNE))

test_dataset = (test_dataset
                .map(preprocess_for_vgg16, num_parallel_calls=AUTOTUNE)
                .cache()
                .batch(TRAIN_BATCH_SIZE)
                .prefetch(AUTOTUNE))

In [5]:
print(tf.__version__)

2.12.0


Training

In [11]:
# Model definition within the strategy scope
with strategy.scope():
    base_model = tf.keras.applications.VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
    base_model.trainable = False

    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=INITIAL_LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)
reduce_lr_on_plateau = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=6, verbose=1, min_lr=1e-8)

# Training
model.fit(train_dataset, epochs=NUM_EPOCHS, validation_data=test_dataset, validation_freq=VAL_FREQ)

model.save('flower_model')


Epoch 1/2000


2023-10-04 22:17:43.940107: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-10-04 22:17:44.053212: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


1/1 [==============================] - 10s 10s/step - loss: 14.8637 - accuracy: 0.0118
Epoch 2/2000
1/1 [==============================] - 1s 571ms/step - loss: 12.5578 - accuracy: 0.0098
Epoch 3/2000
1/1 [==============================] - 1s 668ms/step - loss: 11.2589 - accuracy: 0.0157
Epoch 4/2000
1/1 [==============================] - 1s 649ms/step - loss: 10.4170 - accuracy: 0.0186
Epoch 5/2000
1/1 [==============================] - 1s 638ms/step - loss: 9.7755 - accuracy: 0.0225
Epoch 6/2000
1/1 [==============================] - 1s 680ms/step - loss: 9.2505 - accuracy: 0.0275
Epoch 7/2000
1/1 [==============================] - 1s 655ms/step - loss: 8.8088 - accuracy: 0.0304
Epoch 8/2000
1/1 [==============================] - 1s 652ms/step - loss: 8.4283 - accuracy: 0.0333
Epoch 9/2000
1/1 [==============================] - 1s 667ms/step - loss: 8.0929 - accuracy: 0.0343
Epoch 10/2000
1/1 [==============================] - 1s 656ms/step - loss: 7.7919 - accuracy: 0.0353
Epoch 11/

2023-10-04 22:21:39.953449: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-10-04 22:21:40.059573: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


1/1 [==============================] - 20s 20s/step - loss: 0.4981 - accuracy: 0.9176 - val_loss: 2.1068 - val_accuracy: 0.5040
Epoch 201/2000
1/1 [==============================] - 1s 596ms/step - loss: 0.4945 - accuracy: 0.9176
Epoch 202/2000
1/1 [==============================] - 1s 675ms/step - loss: 0.4909 - accuracy: 0.9196
Epoch 203/2000
1/1 [==============================] - 1s 672ms/step - loss: 0.4874 - accuracy: 0.9186
Epoch 204/2000
1/1 [==============================] - 1s 685ms/step - loss: 0.4841 - accuracy: 0.9206
Epoch 205/2000
1/1 [==============================] - 1s 664ms/step - loss: 0.4807 - accuracy: 0.9216
Epoch 206/2000
1/1 [==============================] - 1s 574ms/step - loss: 0.4772 - accuracy: 0.9216
Epoch 207/2000
1/1 [==============================] - 1s 666ms/step - loss: 0.4739 - accuracy: 0.9216
Epoch 208/2000
1/1 [==============================] - 1s 638ms/step - loss: 0.4706 - accuracy: 0.9235
Epoch 209/2000
1/1 [==============================] - 1s